## <center><u><span style="color:orange">ChatGPT</span> Cover Letter Writer</u></center>
### Because what good is a resume optimizer to speed up the application process... if you have to keep writing a cover letter to go along with it?
#### <b><u><span style="color:red">NB</span>:</u></b> 
#### I seriously debated doing this because I used to think cover letters were where you could shine as an applicant. You know, <span style="color:blue">"I'm perfect for this job because I'm cooler under pressure than a diamond Lamborghini."</span> That kind of thing. 
#### Sadly, I don't think that's the case anymore. Too many applicants fighting over too few jobs - Recruiters just don't have time to read all of those things.
#### <b><span style="color:green">BUT</span></b> for those resumes that <b><i>do</b></i> crack the ATS programs, having a tailored cover letter couldn't hurt, and with chat utilizing each individual's resume, each of those cover letters will be unique.
#### <u>So here are the goals of this notebook</u>:
> ##### 1.) Take in a job description (just like the <b><span style="color:orange">ChatGPT Resume Optimizer</span></b>);
> ##### 2.) Prompt <b><span style="color:orange">ChatGPT</span></b> to write a cover letter for that job using experiences on your resume; and
> ##### 3.) Adding this to our <b><span style="color:gold">Gradio</span></b> UI for a One Stop Shop for all your resume needs.
#### Let's get to it...
***
#### <span style="color:blue">Step 1: Imports
> ##### You're gonna need a few tools.
***

In [10]:
import os # use personal openai key
import uuid # for unique file usernames
import re # for future 'generate' button
from dotenv import load_dotenv # loads environmental variables

# for working with OpenAI:
from openai import OpenAI

# HTML to Markdown for editing
from markdown import markdown
from IPython.display import display, Markdown # <-- make it look nice in the notebook

# for design
from weasyprint import HTML # see previous notebook for weasyprint notes

***
#### <span style="color:blue">Step 2: Check To See If '.env' File Data Loaded</span>
> ##### Returning 'True' means your environment variables (api keys, etc.) have been discretely loaded.
***

In [3]:
load_dotenv()

True

***
#### <span style="color:blue">Step 3: Create A Directory To Store Your Cover Letters</span>
> ##### Gotta give your cover letters a home.
***

In [4]:
os.makedirs("gpt_cover_letters", exist_ok=True)

***
#### <span style="color:blue">Step 4: Read In Your Markdown Resume
> ##### For this notebook only. The final UI will use one resume window for all its functionality.
***

In [7]:
file_path = input("Please paste the full path to your resume file (eg: '/Users/your_name/Desktop/your_resume.md'): ").strip()

with open(file_path, "r", encoding = "utf-8") as resume_file:
    resume_string = resume_file.read()

Please paste the full path to your resume file (eg: '/Users/your_name/Desktop/your_resume.md'):  /Users/nicholasjoseph/Desktop/FailedResumes/nj_master_resume.md


***
#### <span style="color:blue">Step 5: Paste In the Job Description</span>
> ##### Also for this or this notebook only. The final UI will use one job description window for all its functionality.
***

In [8]:
jd_string = input("Please paste selected job descripton here: ")

Please paste selected job descripton here:  About the job The simple task of buying software, services, or tools at work has become hopelessly complicated at even the most innovative companies in the world. Today, enterprises spend $120T+ per year globally (>30 times larger than annual consumer e-commerce spend) and rely on vendors more than ever before to run their businesses.  Our cofounders started Zip in 2020 to address this seemingly intractable problem with a purpose-built procurement platform that provides a simple, consumer-grade user experience. Within the last 4 years, Zip has created a new category and developed the leading solution in this $50B+ TAM space. Today, the world’s leading companies like OpenAI, Snowflake, Anthropic, Coinbase, and Prudential rely on Zip to manage billions of dollars in spend.  We have a world-class team coming from category-defining companies like Airbnb, Meta, Stripe, Salesforce, Apple, and Google. With a $2.2 billion valuation and $370 million i

***
#### <span style="color:blue">Step 6: Input and Slug (Clean / Standardize) the Company Name 
> ##### Dumps all the stuff like spaces and apostrophes so you save 'olemissrebels.pdf' and not 'Ole Miss Rebels.pdf.' Spaces are not your friend.
***

In [11]:
# strip whitespace 
company_raw = input("Please enter the company's name: ").strip()
# cut out anything that's not a letter or number from the company name and lowercase everything
company_slugged = re.sub(r"[^a-zA-Z0-9_-]", "", company_raw.lower())

Please enter the company's name:  Zip


***
#### <span style="color:blue">Step 7: Template Some Prompt Engineering</span>
> ##### Here's where you tell <b><span style="color:orange">ChatGPT</span></b> what kind of cover letter you want. For this notebook and application, we're going for professional but familiar tone.
***

In [13]:
prompt_template = lambda resume_string, job_desc_string : f"""
### Role
You are a professional cover letter writer tasked with creating a compelling, one-page cover letter that clearly shows 
how my background and experience meet — or exceed — the requirements of a specific job description.

The tone should be professional yet conversational — confident without arrogance, polished but human. 

Your goal is to help me stand out by aligning the most relevant parts of my resume with the needs of the role.

---

### Guidelines

1. **Relevance**
   - Highlight the 2–3 most relevant roles or experiences that directly support the job description.
   - Cut any content that doesn't contribute to a concise, tailored message.
   - Prioritize transferable skills, project outcomes, and business impact over duties.

2. **Action & Results**
   - Use strong action verbs and quantify results when possible (e.g., % growth, time saved, revenue impact).

3. **ATS Alignment**
   - Naturally weave in job description keywords and phrasing to optimize for applicant tracking systems (ATS).

4. **Tone**
   - Aim for confident, plainspoken language — not overly formal or robotic.
   - Speak directly to the reader, like a capable peer making their case.

5. **Formatting**
   - Return the full cover letter in **clean Markdown format** with appropriate paragraph spacing for readability.

---

### Input

**My Resume:**
{resume_string}

**Job Description:**
{job_desc_string}

---

### Output

A complete, one-page **Markdown-formatted cover letter** that:
- Emphasizes the most relevant experience, skills, and accomplishments;
- Reflects the language and priorities of the job description;
- Shows confidence, competence, and personality without fluff.
"""

# assign it a variable for use later on in the code
prompt = prompt_template(resume_string, jd_string)

***
#### <span style="color:blue">Step 8: Generate the Cover Letter with <span style="color:darkorchid">ChatGPT</span>'s <span style="color:purple">gpt-4o-mini</span> engine.
> ##### Again, please see <span style="color:firebrick">res_optimizer_chatgpt</span> notebook for details, but here is where you call <span style="color:green">OpenAI</span> and structure the response you want.

In [14]:
# set up client
open_apikey = os.environ.get("openapi_apikey")
    
client = OpenAI(api_key = open_apikey)

response = client.chat.completions.create(
    model = "gpt-4o-mini",
    
    messages = [
        {"role" : "system", "content" : "Expert Cover Letter Writer"},
        {"role" : "user", "content" : prompt}
    ],
     
    temperature = 0.7 #<-- standard. tried a level of 0.8, and things got wonky.
)

# Get our response
response_string = response.choices[0].message.content

***
#### <span style="color:blue">Step 9: Review the Cover Letter</span> <span style="color:darkorchid">ChatGPT</span> <span style="color:blue">Came Up With</span>
> ##### A good place to check for spelling and grammatical errors, as well as the (<i>cough, cough</i>) "dubiousness" of experience elaboration.
***

In [15]:
display(Markdown(response_string))

```markdown
# Nicholas Joseph  
[nickpjoseph210@gmail.com](mailto:nickpjoseph210@gmail.com) | (210) 771-9853 | [LinkedIn](https://www.linkedin.com/in/nickjoseph) | [GitHub](https://github.com/npj210mlk)  
[Date]

[Hiring Manager's Name]  
Zip  
[Company Address]  
[City, State, Zip]  

Dear [Hiring Manager's Name],

I am excited to apply for the Solution Engineer position at Zip, as advertised. With over 15 years of experience in business development and a strong technical background in data engineering, I possess a unique blend of skills that position me to contribute effectively to your mission of simplifying procurement through innovative technology.

In my recent role as a Cloud Data Engineer at Data Clymer, I collaborated closely with cross-functional teams, fostering strong relationships to deliver high-impact solutions. I developed scalable data pipelines for projects like the NFL's ‘Fan 360’ initiative, reducing source data collection times from 24 hours to just 15 minutes—an impressive 98.9% improvement. This experience honed my ability to articulate the value of technical solutions in addressing customer pain points, a key requirement for the Solution Engineer role at Zip.

Additionally, while serving as Project Lead at Noble Services, I successfully managed multiple projects with a total value under $1M, achieving zero overages while maintaining a 100% worker safety rate. This success not only resulted in five-star ratings across customer review platforms but also involved translating complex client requirements into actionable project plans. My proficiency in leading discovery sessions and troubleshooting technical issues will be invaluable in helping prospective customers envision how Zip’s solutions can seamlessly integrate into their procurement landscape.

I am particularly drawn to this role because of Zip’s commitment to driving value for its customers, and I am eager to leverage my solution selling skills and technical acumen to help prospective clients navigate their procurement challenges. I pride myself on being a self-starter, quickly grasping new technologies, and providing insightful feedback to enhance product offerings—skills that align perfectly with the innovative culture at Zip.

I look forward to the opportunity to discuss how my experience and vision can align with the exciting future at Zip. Thank you for considering my application. I hope to contribute to your team as we redefine the procurement experience for leading enterprises.

Warm regards,

Nicholas Joseph
```


***
#### <span style="color:blue">Step 10: Happy? Let's Go Ahead and Save It As A PDF Using <span style="color:deeppink">Weasyprint</span> and that '<span style="color:green">uuid</span>' Library We Imported Earlier 
> ##### <span style="color:deeppink">Weasyprint</span> basically helps you turn HTML and Markdown files into PDFs. It's got its problems, but it's free and pretty powerful.
> ##### Again, <span style="color:green">uuid</span> lets us generate unique usernames to help make tracking our cover letters easier.

In [16]:
# get an 8-character unique id from uuid
unique_id = str(uuid.uuid4())[:8]
# Save and send files to the gpt_cover_letters directory we created in Step 3.
output_pdf_file = f"gpt_cover_letters/{company_slugged}{unique_id}_optimized.pdf"
output_markdown_file = f"gpt_cover_letters/{company_slugged}{unique_id}_optimized.md"

In [17]:
# Convert / save response to HTML so the web can read it using Weasyprint
html_content = markdown(response_string)
HTML(string=html_content).write_pdf(output_pdf_file)

***
#### <span style="color:blue">Step 11: Want To Make Changes? Write and Save This Letter As A Markdown File</span>
> ##### Markdown is awesome - you can edit text and do some pretty powerful things with code so easy knuckle-draggers like me can understand.
***

In [18]:
with open(output_markdown_file, "w", encoding = "utf-8") as file:
    file.write(response_string)